In [ ]:
# ============================================================
#   FITNESS SENTIMENT ANALYSIS MODEL
#   Model  : cardiffnlp/twitter-roberta-base-sentiment
#   Purpose: Analyse gym/fitness text → recommend workout
#            & diet plan based on detected sentiment
# ============================================================

import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
# ────────────────────────────────────────────────────────────
#  1.  LOAD MODEL ONCE  (import-time, not per-call)
# ────────────────────────────────────────────────────────────

ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer     = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model         = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)
LABELS        = ["Negative", "Neutral", "Positive"]
 # ────────────────────────────────────────────────────────────
#  INITIALIZE API & LOAD MODEL ONCE
# ────────────────────────────────────────────────────────────

app = FastAPI(
    title="Fitness Sentiment Analysis API",
    description="Analyzes fitness text and returns sentiment-based workout and diet plans."
)

ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer     = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model         = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)
LABELS        = ["Negative", "Neutral", "Positive"]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# ────────────────────────────────────────────────────────────
#  2.  PRE-PROCESSING DICTIONARIES
# ────────────────────────────────────────────────────────────


In [ ]:
# --- 2a. Emoji → sentiment-descriptive text ------------------
EMOJI_MAP = {
    "💪": " strong motivated energetic ",
    "🔥": " on fire pumped energetic ",
    "😤": " determined aggressive motivated ",
    "🏋️": " lifting weights training hard ",
    "🏆": " champion winning achieving goal ",
    "🎯": " focused goal crushing it ",
    "⚡": " powerful energetic explosive ",
    "😊": " happy positive feeling good ",
    "😁": " very happy excited positive ",
    "😃": " excited enthusiastic positive ",
    "😍": " loving it very positive ",
    "🤩": " amazing super excited positive ",
    "👍": " good positive approved ",
    "❤️":  " love positive happy ",
    "🥳": " celebrating winning positive ",
    "😒": " disappointed unhappy negative ",
    "😔": " sad down feeling low ",
    "😩": " exhausted drained very tired ",
    "😫": " depleted burnt out negative ",
    "😢": " sad upset emotional negative ",
    "😭": " very sad crying very negative ",
    "😴": " sleepy tired low energy ",
    "🥱": " bored lazy unmotivated ",
    "🤒": " sick ill cannot train ",
    "🤮": " very sick unwell negative ",
    "💔": " heartbroken sad demotivated ",
    "👎": " bad negative disapprove ",
    "😐": " okay neutral normal average ",
    "😑": " blank neutral indifferent ",
}

# --- 2b. Gym / Fitness Slang → expanded meaning --------------
GYM_SLANG = {
    # High-energy / Positive
    "beast mode"    : "extremely motivated training at maximum intensity",
    "beastmode"     : "extremely motivated training at maximum intensity",
    "killing it"    : "performing excellently great progress",
    "crushing it"   : "performing excellently great progress",
    "on fire"       : "feeling energetic performing great",
    "new pr"        : "achieved personal record great accomplishment",
    "new pb"        : "achieved personal best great accomplishment",
    "hit a pr"      : "achieved personal record very happy",
    "hit a pb"      : "achieved personal best very happy",
    "pr"            : "personal record achievement",
    "pb"            : "personal best achievement",
    "gains"         : "muscle progress improvement positive",
    "gainz"         : "muscle progress improvement positive",
    "swole"         : "very muscular strong positive",
    "jacked"        : "very muscular strong positive",
    "shredded"      : "very lean and fit positive",
    "ripped"        : "very lean muscular positive",
    "pump"          : "great muscle pump feeling energized",
    "sweat session" : "intense workout very positive",
    "grind mode"    : "working hard consistently motivated",
    "hustle"        : "working hard motivated positive",
    "no days off"   : "training every day very motivated",
    "leg day"       : "leg training day workout",
    "push day"      : "push muscle group training",
    "pull day"      : "pull muscle group training",
    "hiit"          : "high intensity interval training",
    "bulk"          : "muscle building eating more phase",
    "bulking"       : "muscle building eating more phase",
    "lean bulk"     : "clean muscle building controlled diet",
    "cut"           : "fat loss diet phase",
    "cutting"       : "fat loss diet phase",
    "recomp"        : "muscle recomposition body transformation",
    "macros"        : "macronutrients balanced nutrition",
    "macro"         : "macronutrient balanced nutrition",
    "cheat meal"    : "planned break from diet okay",
    "cheat day"     : "planned diet break acceptable",
    "refeed"        : "planned high carbohydrate recovery day",
    "natty"         : "natural training no performance drugs",
    "compound"      : "compound movement heavy exercise",
    "superset"      : "back to back exercise technique",
    "dropset"       : "drop set advanced technique",
    "burnout set"   : "high rep burnout training hard",

    # Low-energy / Negative
    "skipping gym"  : "missing workout not going to gym",
    "missed gym"    : "skipped workout session negative",
    "no motivation" : "unmotivated depressed not feeling gym",
    "burnt out"     : "exhausted overtrained very fatigued",
    "overtrained"   : "too much training body exhausted",
    "overtraining"  : "too much training body exhausted",
    "injured"       : "hurt injured unable to train",
    "injury"        : "hurt injured pain unable to train",
    "sore af"       : "extremely sore muscle soreness pain",
    "doms"          : "delayed onset muscle soreness recovery needed",
    "plateau"       : "stuck no progress frustrated",
    "binge"         : "ate too much unhealthy food negative",
    "binging"       : "eating too much unhealthy negative",
    "cheat week"    : "whole week off diet negative",
    "deload"        : "rest recovery reduced training week",
    "rest day"      : "planned rest recovery day",

    # Neutral / Maintenance
    "bulk season"   : "muscle building phase training",
    "gym rat"       : "dedicated gym person consistent",
    "cardio"        : "cardiovascular exercise training",
    "fasted cardio" : "morning empty stomach cardiovascular training",
    "pre workout"   : "pre workout energy supplement training",
    "post workout"  : "post workout recovery nutrition",
    "protein shake" : "protein supplement recovery drink",
    "whey"          : "whey protein supplement",
    "creatine"      : "creatine strength supplement",
    "reps"          : "repetitions exercise",
    "sets"          : "training sets",
}

# --- 2c. Roman Urdu (Pakistani gym / daily slang) -----------
ROMAN_URDU = {
    # Positive
    "mast workout"      : "great excellent workout feeling positive",
    "mast training"     : "great excellent training feeling positive",
    "maza aa gaya"      : "enjoyed it feeling great very positive",
    "achi feeling"      : "feeling good positive satisfied",
    "bohat acha"        : "very good feeling positive",
    "josh aa gaya"      : "feeling pumped energized very motivated",
    "motivated hun"     : "feeling very motivated ready to train",
    "motivated hoon"    : "feeling very motivated ready to train",
    "jeet gaya"         : "won achieved feeling great positive",
    "jeet gayi"         : "won achieved feeling great positive",
    "mein karunga"      : "i will do it determined motivated",
    "mein karonga"      : "i will do it determined motivated",
    "bohat khushi"      : "very happy feeling great positive",
    "khushi hai"        : "feeling happy positive",
    "full josh"         : "full energy very motivated ready",
    "full energy"       : "very energetic motivated ready to train",
    "game on"           : "ready determined very motivated",
    "uthao weights"     : "lift weights training motivated",

    # Negative
    "thak gaya"         : "feeling very exhausted tired cannot train",
    "thak gayi"         : "feeling very exhausted tired cannot train",
    "bahut thaka"       : "extremely tired exhausted drained",
    "bahut thaki"       : "extremely tired exhausted drained",
    "bohat thaka"       : "extremely tired exhausted drained",
    "body sore hai"     : "body is sore aching recovery needed",
    "dard ho raha"      : "body is paining sore uncomfortable",
    "dard ho rahi"      : "body is paining sore uncomfortable",
    "bura lag raha"     : "feeling bad negative low mood",
    "bura lag rahi"     : "feeling bad negative low mood",
    "demotivated hun"   : "feeling demotivated negative not motivated",
    "demotivated hoon"  : "feeling demotivated negative not motivated",
    "kuch nahi karna"   : "do not want to do anything very low motivation",
    "gym nahi gaya"     : "missed gym skipped workout negative",
    "gym nahi gayi"     : "missed gym skipped workout negative",
    "gym nahi jaon ga"  : "not going to gym skipping workout",
    "gym nahi jaon gi"  : "not going to gym skipping workout",
    "haar gaya"         : "lost defeated feeling negative disappointed",
    "haar gayi"         : "lost defeated feeling negative disappointed",
    "nirasha"           : "disappointed hopeless negative",
    "bohat bura"        : "feeling very bad very negative",
    "bilkul man nahi"   : "absolutely no motivation very negative",
    "uthne ka man nahi" : "do not want to get up lazy negative",

    # ── "gy" endings (colloquial Pakistani — gender-neutral past tense) ──
    # e.g. "haar gy" = "we/they lost"  |  "jeet gy" = "we/they won"
    # These differ from formal "gaya/gayi" but are extremely common in chat
    "haar gy"           : "lost defeated feeling negative disappointed",
    "jeet gy"           : "won achieved feeling great positive",
    "thak gy"           : "feeling very exhausted tired cannot train",
    "uth gy"            : "got up started feeling okay",
    "gir gy"            : "fell down failed feeling negative",
    "ruk gy"            : "stopped gave up feeling negative",
    "bhaag gy"          : "ran away escaped okay",
    "gym nahi gy"       : "missed gym skipped workout negative",
    "aa gy"             : "came arrived feeling okay",
    "kr gy"             : "did it accomplished feeling positive",
    "kar gy"            : "did it accomplished feeling positive",
    "ho gy"             : "it happened done okay neutral",
    "ho gya"            : "it happened done okay neutral",
    "ho gyi"            : "it happened done okay neutral",
    "mar gy"            : "exhausted totally drained very negative",

    # Filler / Neutral connectors
    "yr"                : "friend",
    "yaar"              : "friend",
    "bhai"              : "brother friend",
    "yar"               : "friend",
    "kal"               : "yesterday tomorrow",
}

"""
    Full pre-processing pipeline for fitness / gym text.

    Order matters:
      1  Emoji → descriptive phrase   (before lowercasing so Unicode matches)
      2  Lowercase
      3  Hashtag splitting            (#BeastMode → beast mode)
      4  Roman Urdu expansion
      5  Gym / fitness slang expansion
      6  Repeated-character collapse  (sooore → soore)
      7  @mention / URL normalisation (RoBERTa convention)
      8  Whitespace cleanup
    """
    #  3.  PRE-PROCESSING PIPELINE
          Checked BEFORE the model runs.
#Catches phrases like "feeling neutral" / "meh" / "okay aaj" & that the model tends to misread.
# Each entry is a (regex_pattern, forced_label) pair.
# Patterns are matched case-insensitively on the ORIGINAL (raw) text.
# The first match wins — order from most-specific to least-specific.

In [ ]:
EXPLICIT_OVERRIDES = [

    # ── Explicit Negative overrides ──────────────────────────
    (r'\b(no motivation|zero motivation|lost all motivation|can\'t train|cant train)\b',
     "Negative"),
    (r'\b(feel(ing)?\s+(terrible|awful|horrible|miserable|depressed|crushed|broken))\b',
     "Negative"),
    (r'\b(worst day|bad day|really bad|very bad|feeling bad)\b',
     "Negative"),

    # ── Explicit Neutral overrides ────────────────────────────
    (r'\b(feel(ing)?\s+neutral)\b',                         "Neutral"),
    (r'\b(just\s+okay|just\s+ok|feeling\s+ok(ay)?)\b',     "Neutral"),
    (r'\b(meh|average day|average today|so[\s-]so|nothing special)\b',
     "Neutral"),
    (r'\b(okay\s+aaj|thek\s+hai|theek\s+hai|normal\s+day|bas\s+theek)\b',
     "Neutral"),
    (r'\b(not\s+great\s+not\s+bad|not\s+bad\s+not\s+great)\b',
     "Neutral"),
    (r'\b(maintenance\s+(mode|day)|steady\s+state|cruise\s+control)\b',
     "Neutral"),

    # ── Explicit Positive overrides ──────────────────────────
    (r'\b(best day|greatest day|absolutely amazing|on top of the world)\b',
     "Positive"),
]


def _check_explicit_override(text: str):
    """
    Scan raw text for explicit sentiment phrases.
    Returns forced label string if matched, else None.
    """
    for pattern, label in EXPLICIT_OVERRIDES:
        if re.search(pattern, text, flags=re.IGNORECASE):
            return label
    return None



def _replace_emojis(text: str) -> str:
    """Swap each emoji for its sentiment-descriptive phrase."""
    for emoji, phrase in EMOJI_MAP.items():
        text = text.replace(emoji, phrase)
    return text


def _expand_slang(text: str, slang_dict: dict) -> str:
    """
    Replace every slang key in the text with its expansion.
    Sorted longest-first so multi-word phrases match before
    shorter overlapping sub-strings.
    """
    for key, expansion in sorted(slang_dict.items(),
                                  key=lambda x: len(x[0]), reverse=True):
        # Word-boundary aware replacement (case-insensitive)
        pattern = r'(?<!\w)' + re.escape(key) + r'(?!\w)'
        text = re.sub(pattern, f' {expansion} ', text, flags=re.IGNORECASE)
    return text


def _clean_hashtags(text: str) -> str:
    """#BeastMode → beast mode  (split camel-case, drop the #)."""
    def split_hashtag(match):
        tag = match.group(1)
        # Insert space before every uppercase run following a lowercase letter
        spaced = re.sub(r'([a-z])([A-Z])', r'\1 \2', tag)
        return ' ' + spaced.lower() + ' '
    return re.sub(r'#(\w+)', split_hashtag, text)


def _normalize_repeated_chars(text: str) -> str:
    """sooooore → soore  (collapse any char repeated >2 times to 2)."""
    return re.sub(r'(.)\1{2,}', r'\1\1', text)


def _handle_mentions_and_urls(text: str) -> str:
    """RoBERTa training convention: @handle → @user, URLs → http."""
    tokens = text.split()
    cleaned = []
    for token in tokens:
        if token.startswith('@') and len(token) > 1:
            cleaned.append('@user')
        elif token.startswith('http') or token.startswith('www.'):
            cleaned.append('http')
        else:
            cleaned.append(token)
    return ' '.join(cleaned)


def preprocess(text: str) -> str:
    """
    Full pre-processing pipeline for fitness / gym text.

    Order matters:
      1  Emoji → descriptive phrase   (before lowercasing so Unicode matches)
      2  Lowercase
      3  Hashtag splitting            (#BeastMode → beast mode)
      4  Roman Urdu expansion
      5  Gym / fitness slang expansion
      6  Repeated-character collapse  (sooore → soore)
      7  @mention / URL normalisation (RoBERTa convention)
      8  Whitespace cleanup
    """
    text = _replace_emojis(text)             # Step 1
    text = text.lower()                      # Step 2
    text = _clean_hashtags(text)             # Step 3
    text = _expand_slang(text, ROMAN_URDU)   # Step 4
    text = _expand_slang(text, GYM_SLANG)    # Step 5
    text = _normalize_repeated_chars(text)   # Step 6
    text = _handle_mentions_and_urls(text)   # Step 7
    text = re.sub(r'\s+', ' ', text).strip() # Step 8
    return text


#  4.  SENTIMENT ANALYSIS


In [ ]:
def get_sentiment(text: str) -> dict:
    """
    Run RoBERTa sentiment analysis on fitness/gym text.

    Returns
    -------
    {
        "original_text"   : str,
        "processed_text"  : str,
        "scores"          : {"Negative": float, "Neutral": float, "Positive": float},
        "dominant"        : "Negative" | "Neutral" | "Positive",
        "confidence_pct"  : float          # 0-100
    }
    """
    processed = preprocess(text)

    # ── Step A: check explicit override BEFORE the model ─────
    forced_label = _check_explicit_override(text)

    # ── Step B: run RoBERTa ───────────────────────────────────
    encoded = tokenizer(
        processed,
        return_tensors='pt',
        truncation=True,
        max_length=512
    )
    output = model(**encoded)

    raw_scores  = output.logits[0].detach().numpy()
    soft_scores = softmax(raw_scores)

    score_dict = {label: round(float(soft_scores[i]) * 100, 2)
                  for i, label in enumerate(LABELS)}

    # ── Step C: apply override if triggered ──────────────────
    if forced_label:
        dominant         = forced_label
        override_applied = True
        # Raw model scores are still stored accurately below
    else:
        dominant         = max(score_dict, key=score_dict.get)
        override_applied = False

    return {
        "original_text"   : text,
        "processed_text"  : processed,
        "scores"          : score_dict,   # raw model scores always preserved
        "dominant"        : dominant,
        "confidence_pct"  : score_dict[dominant],
        "override_applied": override_applied,   # True = explicit phrase won
    }


#  5.  WORKOUT PLANS  (keyed by dominant sentiment)

In [ ]:
WORKOUT_PLANS = {

    "Positive": {
        "mode"       : "💪 Beast Mode — High Intensity",
        "description": "You're fired up. Chase that PR, go heavy, and make today count.",
        "exercises"  : [
            {"exercise": "Barbell Back Squat",        "sets": 5, "reps": "5",      "note": "Attempt a PR if energy is high"},
            {"exercise": "Deadlift",                  "sets": 4, "reps": "6",      "note": "Compound king — max effort today"},
            {"exercise": "Bench Press",               "sets": 4, "reps": "8",      "note": "Add 2.5–5 kg from last session"},
            {"exercise": "Weighted Pull-Ups",         "sets": 4, "reps": "8–10",   "note": "Add weight if bodyweight feels easy"},
            {"exercise": "Overhead Press",            "sets": 3, "reps": "6–8",    "note": "Full lockout at the top"},
            {"exercise": "HIIT Finisher (sprints)",   "sets": 1, "reps": "20 min", "note": "All-out effort — battle ropes or treadmill"},
        ],
        "intensity"  : "High (85–100% 1RM on compounds)",
        "rest_period": "60–90 sec between sets",
        "tips"       : [
            "Keep rest short to stay in the zone.",
            "Prioritise compound lifts — max hormonal response.",
            "Take creatine pre-workout if supplementing.",
            "Record every set — today is a PR-chasing day.",
        ],
    },

    "Neutral": {
        "mode"       : "🏋️ Steady Grind — Standard Training",
        "description": "Feeling average? Good. Champions are built on consistent ordinary days.",
        "exercises"  : [
            {"exercise": "Dumbbell Lunges",       "sets": 3, "reps": "12 each", "note": "Focus on controlled descent"},
            {"exercise": "Lat Pulldown",          "sets": 3, "reps": "12",      "note": "Full range, slow eccentric"},
            {"exercise": "Incline DB Press",      "sets": 3, "reps": "10–12",   "note": "Moderate weight, mind-muscle"},
            {"exercise": "Cable Rows",            "sets": 3, "reps": "12",      "note": "Squeeze at peak contraction"},
            {"exercise": "Dumbbell Shoulder Press","sets": 3, "reps": "12",     "note": "No momentum, strict form"},
            {"exercise": "Treadmill / Zone-2 Cardio","sets":1,"reps": "30 min", "note": "Conversational pace"},
        ],
        "intensity"  : "Moderate (70–80% 1RM)",
        "rest_period": "90–120 sec between sets",
        "tips"       : [
            "Apply 2.5% progressive overload from last week.",
            "Log your weights — small wins add up.",
            "Neutral days build the habit. Show up.",
        ],
    },

    "Negative": {
        "mode"       : "🧘 Recovery Mode — Active Rest",
        "description": "Feeling down or drained. Your body and mind need care. Moving gently IS progress.",
        "exercises"  : [
            {"exercise": "Slow-pace Walking / Light Treadmill", "sets": 1, "reps": "20–30 min", "note": "Easy stroll, no pressure"},
            {"exercise": "Full-Body Yoga / Stretching",         "sets": 1, "reps": "15–20 min", "note": "Deep breathing, hold each stretch 30 sec"},
            {"exercise": "Foam Rolling",                        "sets": 1, "reps": "10–15 min", "note": "Quads, lats, thoracic spine, calves"},
            {"exercise": "Bodyweight Exercises (optional)",     "sets": 2, "reps": "10",        "note": "Push-ups / air squats only if you feel up to it"},
            {"exercise": "Box Breathing / Meditation",          "sets": 1, "reps": "10 min",    "note": "4-4-4-4 breathing — stress & cortisol relief"},
        ],
        "intensity"  : "Low (bodyweight / mobility only)",
        "rest_period": "As needed — no rush",
        "tips"       : [
            "Rest IS training. Skipping heavy work today protects tomorrow.",
            "Prioritise 7–9 hours of sleep tonight.",
            "Even 10 minutes of movement improves mood.",
            "Talk to your gym buddy — social support matters.",
        ],
    },
}

#  6.  DIET PLANS  (keyed by dominant sentiment)

In [ ]:
DIET_PLANS = {

    "Positive": {
        "mode"       : "🍗 Gains Fuel — High-Performance Diet",
        "description": "You're in beast mode — fuel hard, recover harder.",
        "meals"      : [
            {
                "timing": "Pre-Workout  (90 min before)",
                "foods" : ["Oats + banana + honey",
                           "2 boiled eggs",
                           "Black coffee or green tea"],
                "macros": "~500 kcal | P: 25g | C: 70g | F: 10g",
            },
            {
                "timing": "Post-Workout  (within 30 min)",
                "foods" : ["Whey protein shake (1–2 scoops)",
                           "Banana or 4–5 dates  (fast carbs)",
                           "Peanut butter toast"],
                "macros": "~450 kcal | P: 40g | C: 50g | F: 8g",
            },
            {
                "timing": "Lunch",
                "foods" : ["Grilled chicken breast 200g",
                           "Brown rice or sweet potato",
                           "Mixed salad + olive oil"],
                "macros": "~650 kcal | P: 45g | C: 65g | F: 15g",
            },
            {
                "timing": "Dinner",
                "foods" : ["Salmon or tuna 200g",
                           "Quinoa or whole-wheat roti",
                           "Steamed broccoli + spinach"],
                "macros": "~600 kcal | P: 42g | C: 50g | F: 18g",
            },
        ],
        "hydration"    : "3.5–4 L water + electrolytes during workout",
        "supplements"  : ["Creatine 5g/day", "Whey Protein", "Multivitamin"],
        "key_principle": "Aim for 1.8–2.2g protein per kg bodyweight. Time carbs around your workout.",
    },

    "Neutral": {
        "mode"       : "🥗 Balanced Macros — Maintenance Diet",
        "description": "Consistent clean eating is the boring secret nobody talks about.",
        "meals"      : [
            {
                "timing": "Breakfast",
                "foods" : ["3 eggs (whole or 2 whole + 2 whites)",
                           "Whole-wheat toast",
                           "Glass of milk or dahi"],
                "macros": "~400 kcal | P: 28g | C: 40g | F: 12g",
            },
            {
                "timing": "Lunch",
                "foods" : ["Chicken or dal (lentil soup)",
                           "2 whole-wheat rotis or 1 cup rice",
                           "Salad or raita"],
                "macros": "~550 kcal | P: 35g | C: 60g | F: 10g",
            },
            {
                "timing": "Snack",
                "foods" : ["Mixed nuts 30g",
                           "Greek yogurt / plain dahi",
                           "Apple or banana"],
                "macros": "~300 kcal | P: 12g | C: 35g | F: 12g",
            },
            {
                "timing": "Dinner",
                "foods" : ["Baked / grilled chicken or fish",
                           "Vegetable stir-fry",
                           "Brown rice or whole-wheat roti"],
                "macros": "~550 kcal | P: 38g | C: 50g | F: 14g",
            },
        ],
        "hydration"    : "2.5–3 L water daily",
        "supplements"  : ["Whey Protein (if intake is short)", "Multivitamin"],
        "key_principle": "Track macros 2–3 days a week to spot gaps. Variety in vegetables.",
    },

    "Negative": {
        "mode"       : "🌿 Recovery & Comfort — Anti-Inflammatory Diet",
        "description": "Eat to heal. Nourish your body and protect your mood.",
        "meals"      : [
            {
                "timing": "Breakfast",
                "foods" : ["Oatmeal + berries + honey",
                           "Chamomile or ginger tea",
                           "Banana"],
                "macros": "~350 kcal | P: 10g | C: 65g | F: 5g",
            },
            {
                "timing": "Lunch",
                "foods" : ["Warm chicken soup OR dal (lentil)",
                           "Light roti or crackers",
                           "Steamed soft vegetables"],
                "macros": "~450 kcal | P: 28g | C: 50g | F: 10g",
            },
            {
                "timing": "Snack",
                "foods" : ["Dark chocolate 2–3 squares (70%+)",
                           "Mixed nuts",
                           "Herbal tea (no caffeine)"],
                "macros": "~250 kcal | P: 6g | C: 25g | F: 15g",
            },
            {
                "timing": "Dinner",
                "foods" : ["Khichdi (rice + dal — comfort meal)",
                           "Plain dahi / yogurt",
                           "Light kachumber salad"],
                "macros": "~450 kcal | P: 20g | C: 65g | F: 8g",
            },
        ],
        "hydration"    : "3 L water + ginger-honey tea + coconut water",
        "supplements"  : ["Vitamin D (mood support)", "Magnesium (stress & sleep)", "Omega-3"],
        "key_principle": "Dark chocolate → serotonin. Limit caffeine (raises cortisol). "
                         "Anti-inflammatory foods: turmeric, ginger, berries.",
    },
}

#  7.  MOTIVATIONAL MESSAGES

In [ ]:
MESSAGES = {
    "Positive": (
        "🔥 BRO YOU'RE IN BEAST MODE! "
        "Go shatter that PR — make today legendary. Let's get it!"
    ),
    "Neutral": (
        "💪 Steady grind, bhai. "
        "Champions are forged on average days. Stay consistent — the gains are coming."
    ),
    "Negative": (
        "💙 Hey, it's okay. Even GOATs have tough days. "
        "Take care of yourself today — you'll be back stronger. This feeling is temporary."
    ),
}


#  8.  MAIN PUBLIC FUNCTION
    """
    End-to-end pipeline: raw text → sentiment → fitness plan.

    Parameters
    ----------
    text : str
        Any gym/fitness related input — English, Roman Urdu,
        gym slangs, emojis, mixed language.

    Returns
    -------
    dict with keys:
        sentiment       – full sentiment result dict (scores, dominant, confidence)
        workout_plan    – personalised workout plan dict
        diet_plan       – personalised diet plan dict
        message         – motivational message string
    """

# FOR API ONE Single cell


In [ ]:
def analyze_and_recommend(text: str) -> dict:
    """
    End-to-end pipeline: raw text → sentiment → fitness plan.
    """
    sentiment = get_sentiment(text)
    dominant  = sentiment["dominant"]

    return {
        "sentiment"    : sentiment,
        "workout_plan" : WORKOUT_PLANS[dominant],
        "diet_plan"    : DIET_PLANS[dominant],
        "message"      : MESSAGES[dominant],
    }

# Define the TextInput Pydantic model
class TextInput(BaseModel):
    text: str

# ─── PASTE THE NEW CODE RIGHT HERE AT THE BOTTOM OF THE CELL ───
@app.post("/analyze")
def analyze_api(input_data: TextInput):
    # This hooks up your function above to the web for Render
    return analyze_and_recommend(input_data.text)

#  9.  QUICK TEST  (run:  python fitness_sentiment_model.py)

In [ ]:
if __name__ == "__main__":

    test_inputs = [
        # ── Original notebook cases ──────────────────────────
        ("great day! Pakistan won the match",               "Positive"),   # T1 ✅
        ("yr match haar gy !",                              "Negative"),   # T2 ← BUG FIXED (gy dict)
        ("yr match haar gy 😒 !",                           "Negative"),   # T3 ✅

        # ── Fitness-specific ─────────────────────────────────
        ("beast mode activated 💪🔥 hit a new PR on deadlifts today bhai!", "Positive"),  # T4 ✅
        ("thak gaya yaar, bahut thaka, gym nahi jaon ga aaj 😩",            "Negative"),  # T5 ✅
        ("feeling neutral today, just gonna stick to the plan",             "Neutral"),   # T6 ← BUG FIXED (override)
        ("#LegDay done, DOMS is real but gains don't lie 💪",               "Positive"),  # T7 ✅
        ("no motivation at all, demotivated hoon, bura lag raha 😔",        "Negative"),  # T8 ✅

        # ── Extra edge cases ─────────────────────────────────
        ("just okay today, meh, nothing special",           "Neutral"),
        ("jeet gy bhai! match jeet gy! 🏆",                 "Positive"),
        ("feeling terrible, worst day ever, cant train",    "Negative"),
    ]

    print("=" * 70)
    print("  FITNESS SENTIMENT MODEL — TEST RUN  (v2 — bugs fixed)")
    print("=" * 70)

    passed = 0
    for idx, (inp, expected) in enumerate(test_inputs, 1):
        result   = analyze_and_recommend(inp)
        s        = result["sentiment"]
        got      = s["dominant"]
        override = "⚠ override" if s["override_applied"] else "model"
        ok       = "✅" if got == expected else "❌"
        if got == expected:
            passed += 1

        print(f"\n[T{idx}] {ok}  [{override}]")
        print(f"  Input     : {inp}")
        print(f"  Processed : {s['processed_text'][:80]}{'...' if len(s['processed_text'])>80 else ''}")
        print(f"  Scores    : Neg={s['scores']['Negative']:.1f}%  "
              f"Neu={s['scores']['Neutral']:.1f}%  "
              f"Pos={s['scores']['Positive']:.1f}%")
        print(f"  Dominant  : {got}  (expected: {expected})")
        print(f"  Workout   : {result['workout_plan']['mode']}")
        print(f"  Diet      : {result['diet_plan']['mode']}")
        print("-" * 70)

    print(f"\n  RESULT: {passed}/{len(test_inputs)} passed")

  FITNESS SENTIMENT MODEL — TEST RUN  (v2 — bugs fixed)

[T1] ✅  [model]
  Input     : great day! Pakistan won the match
  Processed : great day! pakistan won the match
  Scores    : Neg=0.2%  Neu=1.4%  Pos=98.4%
  Dominant  : Positive  (expected: Positive)
  Workout   : 💪 Beast Mode — High Intensity
  Diet      : 🍗 Gains Fuel — High-Performance Diet
----------------------------------------------------------------------

[T2] ✅  [model]
  Input     : yr match haar gy !
  Processed : friend match lost defeated feeling negative disappointed !
  Scores    : Neg=93.6%  Neu=5.7%  Pos=0.7%
  Dominant  : Negative  (expected: Negative)
  Workout   : 🧘 Recovery Mode — Active Rest
  Diet      : 🌿 Recovery & Comfort — Anti-Inflammatory Diet
----------------------------------------------------------------------

[T3] ✅  [model]
  Input     : yr match haar gy 😒 !
  Processed : friend match lost defeated feeling negative disappointed disappointed unhappy ne...
  Scores    : Neg=95.5%  Neu=4.0%  Pos=

In [ ]:
# Run this cell to automatically save your code as a clean file for Render
import inspect
import sys

# This automatically writes your Colab notebook code out into a file called app.py
!jupyter nbconvert --to script --output app.py

This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr

In [ ]:
import os

app_code = """
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel

app = FastAPI()

class TextInput(BaseModel):
    text: str

@app.post("/analyze")
def analyze_api(input_data: TextInput):
    return analyze_and_recommend(input_data.text)
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("Successfully created app.py!")

Successfully created app.py!


In [ ]:
requirements_code = """
fastapi
uvicorn
pydantic
transformers
scipy
torch --index-url https://download.pytorch.org/whl/cpu
"""

with open("requirements.txt", "w") as f:
    f.write(requirements_code.strip())

print("Successfully created requirements.txt!")

Successfully created requirements.txt!


In [ ]:
%%writefile app.py
# ============================================================
#   FITNESS SENTIMENT ANALYSIS API (FASTAPI)
# ============================================================

import re
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax

# 1. INITIALIZE API & LOAD MODEL
app = FastAPI(title="Fitness Sentiment Analysis API")

ROBERTA_MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer     = AutoTokenizer.from_pretrained(ROBERTA_MODEL)
model         = AutoModelForSequenceClassification.from_pretrained(ROBERTA_MODEL)
LABELS        = ["Negative", "Neutral", "Positive"]

class TextInput(BaseModel):
    text: str

# 2. DICTIONARIES & MAPPINGS
EMOJI_MAP = {
    "💪": " strong motivated energetic ", "🔥": " on fire pumped energetic ",
    "😤": " determined aggressive motivated ", "🏋️": " lifting weights training hard ",
    "😊": " happy positive feeling good ", "😒": " disappointed unhappy negative ",
    "😔": " sad down feeling low ", "😩": " exhausted drained very tired ",
    "😴": " sleepy tired low energy ", "🥱": " bored lazy unmotivated "
}

GYM_SLANG = {
    "beast mode": "extremely motivated training at maximum intensity",
    "killing it": "performing excellently great progress",
    "crushing it": "performing excellently great progress",
    "new pr": "achieved personal record great accomplishment",
    "gains": "muscle progress improvement positive",
    "swole": "very muscular strong positive",
    "shredded": "very lean and fit positive",
    "pump": "great muscle pump feeling energized",
    "no motivation": "unmotivated depressed not feeling gym",
    "burnt out": "exhausted overtrained very fatigued",
    "injured": "hurt injured unable to train",
    "sore af": "extremely sore muscle soreness pain"
}

ROMAN_URDU = {
    "mast workout": "great excellent workout feeling positive",
    "maza aa gaya": "enjoyed it feeling great very positive",
    "achi feeling": "feeling good positive satisfied",
    "josh aa gaya": "feeling pumped energized very motivated",
    "thak gaya": "feeling very exhausted tired cannot train",
    "thak gayi": "feeling very exhausted tired cannot train",
    "body sore hai": "body is sore aching recovery needed",
    "dard ho raha": "body is paining sore uncomfortable",
    "demotivated hun": "feeling demotivated negative not motivated",
    "gym nahi gaya": "missed gym skipped workout negative"
}

EXPLICIT_OVERRIDES = [
    (r'\b(no motivation|zero motivation|lost all motivation|can\'t train)\b', "Negative"),
    (r'\b(just\s+okay|just\s+ok|feeling\s+ok(ay)?)\b', "Neutral"),
    (r'\b(meh|average day|so[\s-]so)\b', "Neutral"),
    (r'\b(best day|greatest day|absolutely amazing)\b', "Positive")
]

WORKOUT_PLANS = {
    "Positive": {
        "mode": "💪 Beast Mode — High Intensity",
        "exercises": [
            {"exercise": "Barbell Back Squat", "sets": 5, "reps": "5", "note": "Heavy effort"},
            {"exercise": "Deadlift", "sets": 4, "reps": "6", "note": "Max effort today"},
            {"exercise": "Bench Press", "sets": 4, "reps": "8", "note": "Push your limits"}
        ]
    },
    "Neutral": {
        "mode": "🏋️ Steady Grind — Standard Training",
        "exercises": [
            {"exercise": "Dumbbell Lunges", "sets": 3, "reps": "12 each", "note": "Controlled form"},
            {"exercise": "Lat Pulldown", "sets": 3, "reps": "12", "note": "Slow execution"},
            {"exercise": "Incline DB Press", "sets": 3, "reps": "10-12", "note": "Focus on the pump"}
        ]
    },
    "Negative": {
        "mode": "🧘 Recovery Mode — Active Rest",
        "exercises": [
            {"exercise": "Light Treadmill / Walk", "sets": 1, "reps": "20-30 min", "note": "Easy pace"},
            {"exercise": "Full-Body Stretching", "sets": 1, "reps": "15 min", "note": "Relax and breathe"},
            {"exercise": "Foam Rolling", "sets": 1, "reps": "10 min", "note": "Target sore muscles"}
        ]
    }
}

DIET_PLANS = {
    "Positive": {
        "mode": "🍗 Gains Fuel — High-Performance Diet",
        "details": "High protein, timed carbs around training window. Focus on chicken, eggs, rice, and oats."
    },
    "Neutral": {
        "mode": "🥗 Balanced Macros — Maintenance Diet",
        "details": "Standard clean macro split. Balanced intake of lentils (dal), whole-wheat roti, chicken, and dahi."
    },
    "Negative": {
        "mode": "🌿 Recovery & Comfort — Anti-Inflammatory Diet",
        "details": "Hydrate deeply. Opt for warm chicken soup, fruits, light lentils, and anti-inflammatory ginger tea."
    }
}

MESSAGES = {
    "Positive": "Aap full josh mein hain! Keep smashing those goals! 🔥",
    "Neutral": "Consistency is key. Bas lage raho, wins will follow. 👍",
    "Negative": "Koi baat nahi, rest is part of the process. Khayal rakhain apni body ka. 🙏"
}

# 3. CORE LOGIC FUNCTIONS
def clean_text(text: str) -> str:
    text_lower = text.lower()
    for emoji, translation in EMOJI_MAP.items():
        text_lower = text_lower.replace(emoji, translation)
    for slang, translation in GYM_SLANG.items():
        text_lower = re.sub(r'\b' + re.escape(slang) + r'\b', translation, text_lower)
    for urdu, translation in ROMAN_URDU.items():
        text_lower = re.sub(r'\b' + re.escape(urdu) + r'\b', translation, text_lower)
    return text_lower

def get_sentiment(text: str) -> dict:
    cleaned = clean_text(text)

    # Check explicit overrides first
    for pattern, label in EXPLICIT_OVERRIDES:
        if re.search(pattern, cleaned):
            return {"dominant": label, "scores": {"Negative": 0.0, "Neutral": 0.0, "Positive": 0.0, label: 1.0}}

    # Fallback to AI Model
    inputs = tokenizer(cleaned, return_tensors="pt")
    outputs = model(**inputs)
    scores = softmax(outputs.logits[0].detach().numpy())

    score_dict = {LABELS[i]: float(scores[i]) for i in range(len(LABELS))}
    dominant_label = max(score_dict, key=score_dict.get)
    return {"dominant": dominant_label, "scores": score_dict}

def analyze_and_recommend(text: str) -> dict:
    sentiment = get_sentiment(text)
    dominant  = sentiment["dominant"]
    return {
        "sentiment"    : sentiment,
        "workout_plan" : WORKOUT_PLANS[dominant],
        "diet_plan"    : DIET_PLANS[dominant],
        "message"      : MESSAGES[dominant],
    }

# 4. WEB ENDPOINT
@app.post("/analyze")
def analyze_api(input_data: TextInput):
    try:
        return analyze_and_recommend(input_data.text)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

Overwriting app.py
